
# Market Basket Recommender
This notebook uses **your dataset** and borrows feature ideas from your notebooks:
- Frequency features: `freq_ui`, `Bu`, `Bui`
- Recency features: `cum_days`, `age_days`, `weight`, `normalized_recency`
- TF–IDF features: `tf`, `idf`, `tfidf_score`
- Ratings-style composite: `ranke_ui` (from normalized frequency/recency/tfidf)

Then it mines **association rules** (support, confidence, lift, interest) and exposes:
```python
recommend(current_basket, k=10, mode="lift")  # also "profit" if margin is available
```


## 0) Setup

In [1]:

# %pip install pandas numpy mlxtend matplotlib
import os, glob, math, itertools
from collections import Counter, defaultdict
from typing import List, Tuple, Dict, Iterable, Optional, Set

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, apriori, association_rules

pd.set_option("display.max_colwidth", 120)
print("Ready.")


Ready.



## 1) Auto-detect and load your data
The loader looks for either:
- **Instacart-style**: `orders.csv`, `order_products__prior.csv`, `products.csv`
- **Generic transactions**: a file with at least `basket_id` (or `order_id`) and `item_id` (or `product_id`).


In [5]:
def load_data():
    orders = pd.read_csv("data/orders.csv")
    order_products = pd.read_csv("data/order_products__prior.csv")
    products = pd.read_csv("data/products.csv")
    return orders, order_products, products

orders, order_products, products = load_data()
print("orders.head():"); display(orders.head(3))
print("order_products.head():"); display(order_products.head(3))
if products is not None:
    print("products.head():"); display(products.head(3))


FileNotFoundError: [Errno 2] No such file or directory: 'data/orders.csv'


## 2) Build baskets and product names
We will try to map `product_id → product_name` if available.


In [ ]:

# Map id → name
name_map = None
for name_col in ["product_name","item_name","name","title","aisle_id"]:
    if products is not None and name_col in products.columns:
        name_map = dict(zip(products["product_id"], products[name_col]))
        break

op = order_products.copy()
if name_map is not None:
    op["product_name"] = op["product_id"].map(name_map)
else:
    op["product_name"] = op["product_id"].astype(str)

# merge minimal order info
ord_cols = ["order_id","user_id","order_number","days_since_prior_order"]
avail = [c for c in ord_cols if c in orders.columns]
orders_min = orders[avail].copy()
data = op.merge(orders_min, on="order_id", how="left")

# Build baskets: one row per order
baskets_df = data.groupby("order_id")["product_name"].apply(list).reset_index()
baskets = baskets_df["product_name"].tolist()
print("Num baskets:", len(baskets), "| unique items:", len(set(e for t in baskets for e in set(t))))



## 3) Features from your notebooks
We create the following user–item features:
- **Frequency:** `freq_ui` = number of times user *u* bought item *i*; `Bu` = baskets per user; `Bui` = baskets where *u* bought *i*.
- **Recency:** compute `cum_days`, `total_days`, `age_days`, exponential `weight`, and `normalized_recency` per (u,i).
- **TF–IDF:** per user *u*, `tf = purchase_count / total_products`; global `idf = log(N_users / (1 + users_who_bought))`; `tfidf_score = tf * idf`.
- **Composite “rating”:** `ranke_ui` from normalized frequency/recency/tfidf (simple weighted sum you can tune).


In [ ]:

# Ensure user_id exists; if not, make a single dummy user
if "user_id" not in data.columns:
    data["user_id"] = 0

# Frequency features
ui = data.groupby(["user_id","product_name"]).size().rename("freq_ui").reset_index()
Bu = data.groupby("user_id")["order_id"].nunique().rename("Bu").reset_index()
Bui = data.groupby(["user_id","product_name"])["order_id"].nunique().rename("Bui").reset_index()
freq = ui.merge(Bui, on=["user_id","product_name"])
freq = freq.merge(Bu, on="user_id")
display(freq.head(5))

# Recency features (if days_since_prior_order exists)
rec_cols = []
if "days_since_prior_order" in data.columns:
    tmp = data[["user_id","order_id","product_name","days_since_prior_order"]].drop_duplicates()
    tmp = tmp.sort_values(["user_id","order_id"])
    tmp["days_since_prior_order"] = tmp["days_since_prior_order"].fillna(0)
    tmp["cum_days"] = tmp.groupby("user_id")["days_since_prior_order"].cumsum()
    total_days = tmp.groupby("user_id")["days_since_prior_order"].sum().rename("total_days").reset_index()
    tmp = tmp.merge(total_days, on="user_id", how="left")
    tmp["age_days"] = tmp["total_days"] - tmp["cum_days"]
    λ = 0.01
    tmp["weight"] = np.exp(-λ * tmp["age_days"])
    rec = tmp.groupby(["user_id","product_name"])["weight"].sum().rename("recency_score").reset_index()
    # normalize per-user 0-1
    rec["normalized_recency"] = rec.groupby("user_id")["recency_score"].transform(lambda s: (s - s.min())/(s.max()-s.min()+1e-12))
    rec_cols = ["recency_score","normalized_recency"]
else:
    rec = pd.DataFrame(columns=["user_id","product_name","recency_score","normalized_recency"])

# TF–IDF features
up_counts = data.groupby(["user_id","product_name"]).size().rename("purchase_count").reset_index()
u_totals = data.groupby("user_id")["product_name"].size().rename("total_products").reset_index()
tf = up_counts.merge(u_totals, on="user_id")
tf["tf"] = tf["purchase_count"] / tf["total_products"]

users_per_item = data.groupby("product_name")["user_id"].nunique().rename("users_who_bought").reset_index()
N_users = data["user_id"].nunique()
idf = users_per_item.copy()
idf["idf"] = np.log(N_users / (1 + idf["users_who_bought"]))

tfidf = tf.merge(idf, on="product_name", how="left")
tfidf["tfidf_score"] = tfidf["tf"] * tfidf["idf"]

# Compose ratings-style score
feats = freq.merge(rec, on=["user_id","product_name"], how="left").merge(tfidf, on=["user_id","product_name"], how="left")
# z-norm per user to avoid domination
for col in ["freq_ui","Bui","Bu","recency_score","normalized_recency","tf","idf","tfidf_score"]:
    if col in feats.columns:
        feats[f"norm_{col}"] = feats.groupby("user_id")[col].transform(lambda s: (s - s.mean())/(s.std()+1e-12))

# composite
feats["ranke_ui"] = (
    0.5 * feats.get("norm_freq_ui", pd.Series(0, index=feats.index)) +
    0.3 * feats.get("norm_normalized_recency", pd.Series(0, index=feats.index)) +
    0.2 * feats.get("norm_tfidf_score", pd.Series(0, index=feats.index))
)
feats.fillna(0, inplace=True)
display(feats.head(8))



## 4) Mine frequent itemsets and rules
We’ll use FP-Growth (or Apriori) and compute support, confidence, lift, and interest.


In [ ]:

transactions = [sorted(list(set(b))) for b in baskets]
te = TransactionEncoder().fit(transactions)
X = pd.DataFrame(te.transform(transactions), columns=te.columns_)

min_support = 0.02 if len(transactions) > 2000 else 0.05
max_len = 3
USE_FPGROWTH = True

if USE_FPGROWTH:
    freq = fpgrowth(X, min_support=min_support, use_colnames=True, max_len=max_len)
else:
    freq = apriori(X, min_support=min_support, use_colnames=True, max_len=max_len)

freq = freq.sort_values("support", ascending=False)
rules = association_rules(freq, metric="confidence", min_threshold=0.1).copy()

# "interest" = confidence - support(consequent)
singletons = freq[freq["itemsets"].apply(lambda s: len(s)==1)][["itemsets","support"]]
cons_support = {list(s)[0]: sup for s, sup in zip(singletons["itemsets"], singletons["support"])}
def consequent_support(cs: frozenset) -> float:
    return cons_support.get(next(iter(cs)), np.nan)
rules["interest"] = rules["confidence"] - rules["consequents"].apply(consequent_support)
rules = rules.sort_values(["lift","confidence","support"], ascending=False)

display(freq.head(10))
display(rules[["antecedents","consequents","support","confidence","lift","interest"]].head(20))



## 5) Recommender function
Given a **current basket** (list of product names), recommend top‑k items using an ensemble of rules.
Scoring: `score = max_{A⊆S} lift(A→j) * confidence(A→j)`, optionally nudged by your `ranke_ui` priors.


In [ ]:

# Pre-index rules by consequent for fast lookup
from functools import lru_cache

rules_idx = []
for _, r in rules.iterrows():
    A = tuple(sorted(list(r["antecedents"])))
    C = list(r["consequents"])[0] if len(r["consequents"])==1 else None
    if C is None: 
        continue
    rules_idx.append((A, C, r["lift"], r["confidence"], r["support"], r["interest"]))

# user-item priors from feats (aggregate across users to get global prior if user unknown)
global_prior = feats.groupby("product_name")["ranke_ui"].mean().rename("prior_score")

@lru_cache(maxsize=10000)
def powerset_cached(items_tuple):
    items = list(items_tuple)
    subs = []
    for m in range(1, min(3, len(items))+1):  # consider subsets up to size 2 to keep it fast
        for comb in itertools.combinations(items, m):
            subs.append(tuple(sorted(comb)))
    return subs

def recommend(current_basket: List[str], k: int=10, use_prior: bool=True) -> pd.DataFrame:
    S = sorted(set(current_basket))
    subsets = powerset_cached(tuple(S))
    best = defaultdict(lambda: 0.0)
    meta = {}
    for A, C, lift_, conf_, supp_, intr_ in rules_idx:
        if set(A).issubset(S):
            score = lift_ * conf_
            if score > best[C]:
                best[C] = score
                meta[C] = {"antecedent": A, "lift": lift_, "confidence": conf_, "support": supp_, "interest": intr_}
    rows = []
    for item, score in best.items():
        if item in S: 
            continue
        base = score
        if use_prior and item in global_prior.index:
            base = base * (1 + 0.1 * max(0.0, global_prior.loc[item]))
        rows.append((item, base, meta[item]))
    recs = pd.DataFrame([(i,s,m["antecedent"],m["lift"],m["confidence"],m["support"],m["interest"]) for i,s,m in rows],
                        columns=["item","score","because","lift","confidence","support","interest"]).sort_values("score", ascending=False).head(k)
    return recs

# Example run using the most common items as a fake current basket
top_items = pd.Series([i for t in transactions for i in t]).value_counts().head(3).index.tolist()
print("Example basket:", top_items)
display(recommend(top_items, k=10))
